<a href="https://colab.research.google.com/github/davidrpugh/introduction-to-deep-learning/blob/master/notebooks/01b-mlp-for-classification-with-pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Layer Perceptrons (MLPs) for Classification with PyTorch

In [ ]:
import numpy as np
from sklearn import compose, datasets, linear_model, metrics, model_selection
from sklearn import pipeline, preprocessing

import torch
from torch import nn, optim


## Binary Classification

### Loading the data

In [ ]:
breast_cancer_dataset = datasets.load_breast_cancer(
    as_frame=True,
)

In [ ]:
print(breast_cancer_dataset["DESCR"])

In [ ]:
breast_cancer_features_df = breast_cancer_dataset["data"]
breast_cancer_target = breast_cancer_dataset["target"]

In [ ]:
breast_cancer_features_df.info()

In [ ]:
breast_cancer_features_df.describe()

In [ ]:
_ = breast_cancer_target.hist()

### Prepare the data

#### Train/Val Split

In [ ]:
RANDOM_STATE = np.random.RandomState(42)


train_features_df, val_features_df, train_target, val_target = (
    model_selection.train_test_split(
        breast_cancer_features_df,
        breast_cancer_target,
        random_state=RANDOM_STATE,
        shuffle=True,
        stratify=breast_cancer_target,
        test_size=0.20,
    )
)


#### Features and target preparation

In [ ]:
def array_to_tensor(arr, dtype=torch.float32):
    return torch.tensor(arr, dtype=dtype)


def series_to_tensor(s, dtype=torch.float32):
    arr = s.to_numpy()
    return array_to_tensor(arr, dtype)


n_samples, _ = train_features_df.shape
prepare_breast_cancer_features = pipeline.make_pipeline(
    preprocessing.QuantileTransformer(
      n_quantiles=n_samples,
      output_distribution="normal",
      random_state=RANDOM_STATE
    ),
    preprocessing.FunctionTransformer(
        func=array_to_tensor
    )
)

prepare_breast_cancer_target = pipeline.make_pipeline(
    preprocessing.FunctionTransformer(
        func=series_to_tensor,
        kw_args={
            "dtype": torch.int64
        }
    )
)

In [ ]:
X_train = prepare_breast_cancer_features.fit_transform(train_features_df)
X_val = prepare_breast_cancer_features.transform(val_features_df)


In [ ]:
print(X_train.shape)
print(X_val.shape)

In [ ]:
y_train = prepare_breast_cancer_target.fit_transform(train_target)
y_val = prepare_breast_cancer_target.transform(val_target)


In [ ]:
# note that targets are now 1 dimensional!
print(y_train.shape)
print(y_val.shape)

### Implementing an MLP for Binary Classification using nn.Sequential

[`nn.Sequential`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html) in PyTorch is a container module that allows for the sequential execution of a series of neural network layers or modules. It simplifies the process of building neural networks with a linear, feed-forward structure by eliminating the need to explicitly define the forward method for each layer.


### Key Characteristics and Use-Cases

* **Ordered Container:** `nn.Sequential` takes a list of `nn.Module` instances (layers) and arranges them in the order they are provided.
* **Automatic Forward Pass:** When an input tensor is passed to an `nn.Sequential` object, it automatically propagates through each contained module in the defined order, with the output of one module serving as the input to the next.
* **Simplified Model Definition:** It offers a concise way to define models, especially for straightforward architectures without complex branching or custom logic within the forward pass.
* **Treat as a Single Module:** The entire `nn.Sequential` container can be treated as a single `nn.Module`, allowing for easy integration into larger models or for applying operations like moving to a device (`.to(device)`) or setting training/evaluation mode (`.train()`, `.eval()`).

In [ ]:
_ = torch.manual_seed(42)

n_features = X_train.size(1)
n_classes = y_train.unique().size(0)


# use the 2/3 heuristic for choosing the number of neurons
n_hidden = (2 * (n_features + n_classes) ) // 3


breast_cancer_model = nn.Sequential(
    nn.Linear(
        in_features=n_features,
        out_features=n_hidden,
        bias=True,
    ),
    nn.ReLU(),
    nn.Linear(
        in_features=n_hidden,
        out_features=n_classes,
        bias=True,
    ),
)

### Loss functions and optimizers

In [ ]:
cross_entropy_loss = nn.CrossEntropyLoss()

sgd = optim.SGD(
    breast_cancer_model.parameters(),
    lr=1e-2,
)

In [ ]:
def train(
    model_fn,
    criterion,
    optimizer,
    X_train,
    y_train,
    X_val,
    y_val,
    n_epochs,
    log_epochs=100,
    ):

    for epoch in range(n_epochs):
        # forward pass
        y_pred = model_fn(X_train)
        train_loss = criterion(y_pred, y_train)

        # backward pass
        train_loss.backward()

        # gradient descent step
        optimizer.step()
        optimizer.zero_grad()

        # evaluate using the validation data
        with torch.no_grad():
            y_pred = model_fn(X_val)
            val_loss = criterion(y_pred, y_val)

        if (epoch + 1) % log_epochs == 0:
            print(f"Epoch {epoch + 1}/{n_epochs}, Training Loss: {train_loss.item(): .4f}, Val Loss: {val_loss.item(): .4f}")


In [ ]:
train(
    breast_cancer_model,
    cross_entropy_loss,
    sgd,
    X_train,
    y_train,
    X_val,
    y_val,
    n_epochs=1000
)

## Multi-class Classification

### Loading the data

In [ ]:
covtype_dataset = datasets.fetch_covtype(
    as_frame=True
)

In [ ]:
print(covtype_dataset["DESCR"])

In [ ]:
covtype_features_df = covtype_dataset["data"]
covtype_target_df = (
    covtype_dataset.get("target")
                   .to_frame()
)

In [ ]:
covtype_features_df.info()

In [ ]:
_ = (
    covtype_target_df.loc[:, "Cover_Type"]
                     .value_counts()
                     .sort_index()
                     .plot(kind="bar")
)

### Preparing the data

#### Train/Val Split

In [ ]:
train_features_df, val_features_df, train_target_df, val_target_df = (
    model_selection.train_test_split(
        covtype_features_df,
        covtype_target_df,
        test_size=0.20,
        shuffle=True,
        stratify=covtype_target_df,
        random_state=RANDOM_STATE
    )
)


#### Features and target preparation

In [ ]:
prepare_covtype_features = pipeline.make_pipeline(
    compose.make_column_transformer(
        (
            "passthrough",
            compose.make_column_selector(
                pattern="^Wilderness_Area_|^Soil_Type_"
            )
        ),
        force_int_remainder_cols=False,
        n_jobs=-1,
        remainder=preprocessing.QuantileTransformer(
            output_distribution="normal",
            random_state=RANDOM_STATE,
        )
    ),
    preprocessing.FunctionTransformer(
        func=array_to_tensor,
    )
)

prepare_covtype_target = pipeline.make_pipeline(
    preprocessing.OrdinalEncoder(
        categories=[
            [1, 2, 3, 4, 5, 6, 7]
        ],
    ),
    preprocessing.FunctionTransformer(
        func=array_to_tensor,
        kw_args={
            "dtype": torch.int64
        }
    ),
    preprocessing.FunctionTransformer(
        func=torch.squeeze,
    )
)



In [ ]:
X_train = prepare_covtype_features.fit_transform(train_features_df)
X_val = prepare_covtype_features.transform(val_features_df)


In [ ]:
print(X_train.shape)
print(X_val.shape)

In [ ]:
y_train = prepare_covtype_target.fit_transform(train_target_df)
y_val = prepare_covtype_target.transform(val_target_df)


In [ ]:
# again note that the targets are 1-dimensional!
print(y_train.shape)
print(y_train.dtype)

print(y_val.shape)
print(y_val.dtype)

### Exercise:

Implement a MLP using `nn.Sequential` that has three hidden layers with sizes 200, 100, and 50. Use `nn.ReLU` activation functions.

In [ ]:
# INSERT YOUR CODE HERE!

### Solution:

In [ ]:
_ = torch.manual_seed(42)

n_features = X_train.size(1)
n_classes = y_train.unique().size(0)

covtype_model = nn.Sequential(
    nn.Linear(
        in_features=n_features,
        out_features=200,
        bias=True,
    ),
    nn.ReLU(),
    nn.Linear(
        in_features=200,
        out_features=100,
        bias=True,
    ),
    nn.ReLU(),
    nn.Linear(
        in_features=100,
        out_features=50,
        bias=True,
    ),
    nn.ReLU(),
    nn.Linear(
        in_features=50,
        out_features=n_classes,
        bias=True,
    ),
)


### Exercise:

Train your MLP for 100 epochs to minimize `nn.CrossEntropyLoss` using plain vanilla `optim.SGD` with a learning rate of 1.

In [ ]:
# INSERT YOUR CODE HERE!

### Solution:

In [ ]:
cross_entropy_loss = nn.CrossEntropyLoss()

sgd = optim.SGD(
    covtype_model.parameters(),
    lr=1e0
)

In [ ]:
train(
    covtype_model,
    cross_entropy_loss,
    sgd,
    X_train,
    y_train,
    X_val,
    y_val,
    n_epochs=100,
    log_epochs=1,
)